In [53]:
import pandas as pd

In [54]:
file_path = "social_media_engagement.csv"
df = pd.read_csv(file_path)
df.head()

,post_id,timestamp,platform,user_id,location,language,text_content,topic_category,toxicity_score,likes_count,shares_count,comments_count,impressions,campaign_name,campaign_phase,user_past_sentiment_avg,user_engagement_growth,buzz_change_rate
0,kcqbs6hxybia,2024-12-09 11:26:15,Facebook,user_52nwb0a6,"Melbourne, Australia",en,Just tried the Chromebook from Google. Best pu...,Pricing,0.0376,5056,2556,3505,18991,WinterWonders,Launch,0.0953,-0.3672,19.1
1,vkmervg4ioos,2024-07-28 19:59:26,Reddit,user_ucryct98,"Tokyo, Japan",ja,Just saw an ad for Microsoft Surface Laptop du...,Delivery,0.9715,3132,3606,2872,52764,BackToSchool,Post-Launch,0.1369,-0.4510,-42.6
2,memhx4o1x6yu,2024-11-23 14:00:12,Twitter,user_7rrev126,"Beijing, China",zh,What's your opinion about Nike's Epic React? ...,Product,0.5124,48402,7050,22505,8887,BlackFriday,Post-Launch,0.2855,-0.4112,17.4
3,bhyo6piijqt9,2024-09-16 04:35:25,Twitter,user_4mxuq0ax,"Lagos, Nigeria",en,Bummed out with my new Diet Pepsi from Pepsi! ...,Delivery,0.4002,18270,2096,14860,6696,FallCollection,Launch,-0.2094,-0.0167,-5.5
4,c9dkiomowakt,2024-09-05 21:03:01,Instagram,user_l1vpox2k,"Berlin, Germany",de,Just tried the Corolla from Toyota. Absolutely...,Product,0.0862,20050,11544,14060,47315,FallCollection,Launch,0.6867,0.0807,38.8


In [55]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12000 entries, 0 to 11999
Data columns (total 18 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   post_id                  12000 non-null  object 
 1   timestamp                12000 non-null  object 
 2   platform                 12000 non-null  object 
 3   user_id                  12000 non-null  object 
 4   location                 12000 non-null  object 
 5   language                 12000 non-null  object 
 6   text_content             12000 non-null  object 
 7   topic_category           12000 non-null  object 
 8   toxicity_score           12000 non-null  float64
 9   likes_count              12000 non-null  int64  
 10  shares_count             12000 non-null  int64  
 11  comments_count           12000 non-null  int64  
 12  impressions              12000 non-null  int64  
 13  campaign_name            12000 non-null  object 
 14  campaign_phase        

In [ ]:
print(df['post_id'].isna().sum())       
print(df['post_id'].duplicated().sum()) 


0
0


### Xử lý thời gian

In [ ]:
df['timestamp'] = pd.to_datetime(df['timestamp'])
df['date'] = df['timestamp'].dt.date
df['time'] = df['timestamp'].dt.time
df['day_of_week'] = df['timestamp'].dt.day_name()
df['month'] = df['timestamp'].dt.month
df['year'] = df['timestamp'].dt.year

print(df[['timestamp', 'date', 'time', 'day_of_week','month','year']].head())

            timestamp        date      time day_of_week  month  year
0 2024-12-09 11:26:15  2024-12-09  11:26:15      Monday     12  2024
1 2024-07-28 19:59:26  2024-07-28  19:59:26      Sunday      7  2024
2 2024-11-23 14:00:12  2024-11-23  14:00:12    Saturday     11  2024
3 2024-09-16 04:35:25  2024-09-16  04:35:25      Monday      9  2024
4 2024-09-05 21:03:01  2024-09-05  21:03:01    Thursday      9  2024


### Xử lý cột location

In [ ]:
df[['city', 'country']] = df['location'].str.split(',', expand=True)
# Xóa cột 'location' vì đã tách thành city và country
df = df.drop(columns=['location'])
# Nếu muốn loại bỏ khoảng trắng thừa ở city và country
df['city'] = df['city'].str.strip()
df['country'] = df['country'].str.strip()
df.loc[df['city'] == 'Singapore', 'country'] = df.loc[df['city'] == 'Singapore', 'country'].fillna('Singapore')
# Kiểm tra kết quả
print(df[['city', 'country']].head(20))

            city       country
0      Melbourne     Australia
1          Tokyo         Japan
2        Beijing         China
3          Lagos       Nigeria
4         Berlin       Germany
5          Seoul   South Korea
6         Madrid         Spain
7      São Paulo        Brazil
8          Milan         Italy
9        Houston           USA
10         Milan         Italy
11  Johannesburg  South Africa
12   Mexico City        Mexico
13       Toronto        Canada
14      Shanghai         China
15   Los Angeles           USA
16          Lyon        France
17     São Paulo        Brazil
18       Houston           USA
19         Paris        France


In [59]:
# Nhóm quốc gia theo khu vực
regions = {
    'North America': ['USA', 'Canada', 'Mexico'],
    'Europe': ['UK', 'Germany', 'France', 'Italy', 'Spain'],
    'Asia': ['China', 'India', 'Japan', 'South Korea', 'Singapore'],
    'Middle East': ['UAE', 'Egypt'],
    'Africa': ['Nigeria', 'South Africa'],
    'South America': ['Brazil'],
    'Oceania': ['Australia']
}

# Tạo dict ánh xạ country → region
country_to_region = {country: region for region, countries in regions.items() for country in countries}

# Gán cột region
df['region'] = df['country'].map(country_to_region)


### Thay thế mã ngôn ngữ bằng tên đầy đủ

In [60]:
language_map = {
    'ar': 'Arabic',
    'de': 'German',
    'en': 'English',
    'es': 'Spanish',
    'fr': 'French',
    'hi': 'Hindi',
    'ja': 'Japanese',
    'pt': 'Portuguese',
    'ru': 'Russian',
    'zh': 'Chinese',
    'ko': 'Korean',
    'it': 'Italian'
}

df['language'] = df['language'].map(language_map).fillna(df['language'])

In [61]:
df.value_counts('language')

language
English       4686
Spanish       1110
Chinese        765
Italian        752
German         748
Portuguese     735
Hindi          734
Arabic         717
French         696
Japanese       695
Korean         362
Name: count, dtype: int64

###  Tách và chuẩn hóa hashtags và mentions từ văn bản.

In [62]:
import re
# Tách hashtags: từ bắt đầu bằng # và theo sau là chữ/số hoặc dấu gạch dưới
df['hashtags'] = df['text_content'].str.findall(r'#\w+')

# Tách mentions: từ bắt đầu bằng @ và theo sau là chữ/số hoặc dấu gạch dưới
df['mentions'] = df['text_content'].str.findall(r'@\w+')

# Chuyển danh sách hashtags và mentions thành chuỗi cách nhau bởi dấu cách
df['hashtags'] = df['hashtags'].apply(lambda x: ', '.join(x))
df['mentions'] = df['mentions'].apply(lambda x: ', '.join(x))
# Kiểm tra kết quả
print(df[['text_content', 'hashtags', 'mentions']].head(20))

                                         text_content  \
0   Just tried the Chromebook from Google. Best pu...   
1   Just saw an ad for Microsoft Surface Laptop du...   
2   What's your opinion about Nike's Epic React?  ...   
3   Bummed out with my new Diet Pepsi from Pepsi! ...   
4   Just tried the Corolla from Toyota. Absolutely...   
5   Nike PowerRelease is subpar! Can't wait to see...   
6   Not sure why with my new Pepsi Wild Cherry fro...   
7   Just saw an ad for Coca-Cola Coke Zero during ...   
8   Just tried the Coke Zero from Coca-Cola. Absol...   
9   My one week review of Google Pixel Watch: Disa...   
10  Just tried the Halo Band from Amazon. Best pur...   
11  Any advice about Google's Pixel Buds? @Marketi...   
12  Pepsi NewYearNewYou is excellent! Can't wait t...   
13  Just unboxed my new Halo Band from Amazon. It'...   
14  Comparing Samsung Galaxy S25 to the competitio...   
15  Just unboxed my new Surface Duo from Microsoft...   
16  Attended the Microsoft Glob

### Trích xuất Brand  Product trong cột text_content.

In [ ]:
# Danh sách các thương hiệu (brands) và sản phẩm (products) phổ biến dựa trên dữ liệu
brands = [
    'Google', 'Microsoft', 'Nike', 'Pepsi', 'Toyota', 'Coca-Cola', 'Amazon', 'Apple', 'Adidas', 'Samsung'
]
brand_mapping = {
    5: 'Google',
    6: 'Microsoft',
    7: 'Nike',
    8: 'Pepsi',
    10: 'Toyota',
    4: 'Coca-Cola',
    2: 'Amazon',
    9: 'Samsung',
    1: 'Adidas',
    3: 'Apple'
}
# Hàm ánh xạ brand_name sang brand_id
def get_brand_id(brand_name):
    for brand_id, name in brand_mapping.items():
        if brand_name == name:
            return brand_id
    return None
# Danh sách sản phẩm phổ biến (có thể mở rộng)
products = [
    'Chromebook', 'Surface Laptop', 'Epic React', 'Diet Pepsi', 'Corolla', 'Coke Zero', 'Halo Band',
    'Pixel Buds', 'Pepsi Wild Cherry', 'Pixel Watch', 'Xbox Series X', 'Coca-Cola Vanilla', 'Pepsi Max',
    'Galaxy S25', 'Surface Duo', 'Nest Hub', 'iMac', 'Ring Camera', 'Air Jordan', 'Yeezy', 'Fire Tablet',
    'Samba', 'NMD', 'FlyKnit', 'Camry', 'Prius', 'Echo Dot', 'Highlander', 'Sienna', 'Vision Pro',
    'Galaxy Tab', 'Eero WiFi', 'Tundra', 'Neo QLED TV', 'Kindle', 'Air Force 1', 'Pixel Tablet',
    'iPad Air', 'Nest Thermostat', 'Ultraboost', 'Predator', 'Stan Smith', 'Air Max', 'Zoom Pegasus',
    'Tacoma', 'Apple Watch', 'AirPods Pro', 'Pixel 8', 'Pepsi Lime', 'Crystal Pepsi', 'Fanta',
    'MacBook Pro', 'Superstar', 'Dri-FIT', 'Coca-Cola Cherry', 'Sprite'
]

# Hàm để trích xuất brand và product từ text_content
def extract_brand_product(text):
    text = str(text)  # Đảm bảo text là chuỗi
    found_brand = None
    found_product = None

    # Tìm brand
    for brand in brands:
        if re.search(r'\b' + re.escape(brand) + r'\b', text, re.IGNORECASE):
            found_brand = brand
            break

    # Tìm product
    for product in products:
        if re.search(r'\b' + re.escape(product) + r'\b', text, re.IGNORECASE):
            found_product = product
            break

    return found_brand, found_product

# Áp dụng hàm để trích xuất brand và product
df[['brand_name', 'product_name']] = df['text_content'].apply(lambda x: pd.Series(extract_brand_product(x)))

# In một số dòng đầu để kiểm tra
print(df[['text_content', 'brand_name', 'product_name']].head(10))

# Tóm tắt số lượng bài đăng theo brand và product
brand_counts = df['brand_name'].value_counts()
product_counts = df['product_name'].value_counts()

                                        text_content brand_name  \
0  Just tried the Chromebook from Google. Best pu...     Google   
1  Just saw an ad for Microsoft Surface Laptop du...  Microsoft   
2  What's your opinion about Nike's Epic React?  ...       Nike   
3  Bummed out with my new Diet Pepsi from Pepsi! ...      Pepsi   
4  Just tried the Corolla from Toyota. Absolutely...     Toyota   
5  Nike PowerRelease is subpar! Can't wait to see...       Nike   
6  Not sure why with my new Pepsi Wild Cherry fro...      Pepsi   
7  Just saw an ad for Coca-Cola Coke Zero during ...  Coca-Cola   
8  Just tried the Coke Zero from Coca-Cola. Absol...  Coca-Cola   
9  My one week review of Google Pixel Watch: Disa...     Google   

        product_name  
0         Chromebook  
1     Surface Laptop  
2         Epic React  
3         Diet Pepsi  
4            Corolla  
5               None  
6  Pepsi Wild Cherry  
7          Coke Zero  
8          Coke Zero  
9        Pixel Watch  


Tạo Campaign_id

In [64]:
# Bảng ánh xạ campaign_id và campaign_name
campaign_mapping = {
    11: 'WinterWonders',
    1: 'BackToSchool',
    2: 'BlackFriday',
    3: 'FallCollection',
    9: 'SummerSale',
    5: 'NewYearNewYou',
    10: 'ValentinesDeals',
    7: 'SpringSale',
    4: 'HolidaySpecial',
    6: 'SpringBlast2025',
    8: 'SummerDreams'
}

# Hàm ánh xạ campaign_name sang campaign_id
def get_campaign_id(campaign_name):
    for campaign_id, name in campaign_mapping.items():
        if campaign_name == name:
            return campaign_id
    return None

### Engagement Rate (Tỷ lệ tương tác)

In [65]:
df["engagement_rate"] = ((df["likes_count"] + df["comments_count"] + df["shares_count"]) / df["impressions"]) * 100

Tạo thêm cột clean_text

In [66]:
import neattext.functions as nfx
import emoji
# 1. Chuyển về chữ thường và giữ emoji
df['clean_text'] = df['text_content'].apply(emoji.demojize).str.lower()

# 2. Loại bỏ user handles
df['clean_text'] = df['text_content'].apply(nfx.remove_userhandles)

# 3. Loại bỏ ký tự đặc biệt (giữ hashtag nếu cần)
df['clean_text'] = df['text_content'].apply(nfx.remove_special_characters)

# 4. Không loại bỏ stopwords để giữ ngữ cảnh cảm xúc
#df['Clean_text'] = df['Clean_Text'].apply(nfx.remove_stopwords)
df.head()

,post_id,timestamp,platform,user_id,language,text_content,topic_category,toxicity_score,likes_count,shares_count,...,year,city,country,region,hashtags,mentions,brand_name,product_name,engagement_rate,clean_text
0,kcqbs6hxybia,2024-12-09 11:26:15,Facebook,user_52nwb0a6,English,Just tried the Chromebook from Google. Best pu...,Pricing,0.0376,5056,2556,...,2024,Melbourne,Australia,Oceania,#Food,,Google,Chromebook,58.538255,Just tried the Chromebook from Google Best pur...
1,vkmervg4ioos,2024-07-28 19:59:26,Reddit,user_ucryct98,Japanese,Just saw an ad for Microsoft Surface Laptop du...,Delivery,0.9715,3132,3606,...,2024,Tokyo,Japan,Asia,"#MustHave, #Food",,Microsoft,Surface Laptop,18.213176,Just saw an ad for Microsoft Surface Laptop du...
2,memhx4o1x6yu,2024-11-23 14:00:12,Twitter,user_7rrev126,Chinese,What's your opinion about Nike's Epic React? ...,Product,0.5124,48402,7050,...,2024,Beijing,China,Asia,"#Promo, #Food, #Trending",,Nike,Epic React,877.202656,Whats your opinion about Nikes Epic React Pro...
3,bhyo6piijqt9,2024-09-16 04:35:25,Twitter,user_4mxuq0ax,English,Bummed out with my new Diet Pepsi from Pepsi! ...,Delivery,0.4002,18270,2096,...,2024,Lagos,Nigeria,Africa,"#Reviews, #Sustainable",,Pepsi,Diet Pepsi,526.075269,Bummed out with my new Diet Pepsi from Pepsi D...
4,c9dkiomowakt,2024-09-05 21:03:01,Instagram,user_l1vpox2k,German,Just tried the Corolla from Toyota. Absolutely...,Product,0.0862,20050,11544,...,2024,Berlin,Germany,Europe,"#Health, #Travel",,Toyota,Corolla,96.489485,Just tried the Corolla from Toyota Absolutely ...


Tạo cột emotion_type (Dùng mô hình dự đoán trên cột clean_text)

In [67]:
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import joblib
import os

# 1. Định nghĩa hàm tải mô hình, tokenizer, và LabelEncoder
def load_model_components(save_dir="./emotion_bert_model"):
    model = AutoModelForSequenceClassification.from_pretrained(save_dir)
    tokenizer = AutoTokenizer.from_pretrained(save_dir)
    le = joblib.load(f"{save_dir}/label_encoder.joblib")
    return model, tokenizer, le

# 2. Hàm dự đoán cảm xúc
def predict_sentiment(text, model, tokenizer, le, max_len=128, device='cpu'):
    if pd.isna(text) or text.strip() == "":
        return "Unknown"
    
    # Chuẩn bị dữ liệu
    encoding = tokenizer(
        text,
        add_special_tokens=True,
        max_length=max_len,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )
    
    # Chuyển dữ liệu sang device
    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)
    
    # Dự đoán
    model.eval()
    with torch.no_grad():
        outputs = model(input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        probs = torch.softmax(logits, dim=1)
        pred_idx = torch.argmax(probs, dim=1).cpu().numpy()[0]
    
    # Chuyển đổi nhãn số thành nhãn gốc
    return le.inverse_transform([pred_idx])[0]

#Tải mô hình, tokenizer, và LabelEncoder
model, tokenizer, le = load_model_components()

#Thiết lập device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

#Dự đoán cảm xúc và tạo cột mới
df['emotion_type'] = df['clean_text'].apply(lambda x: predict_sentiment(x, model, tokenizer, le, device=device))

### Phân Loại Cảm Xúc

In [68]:
# Tạo ánh xạ từ Sentiment sang nhóm polarity
sentiment_to_polarity = {
    "joy": "positive",
    "surprise": "positive",
    "sadness": "negative",
    "anger": "negative",
    "disgust": "negative",
    "fear": "negative",
    "neutral": "neutral"
}

# Thêm cột Emotion_Polarity
df["sentiment_label"] = df["emotion_type"].map(sentiment_to_polarity)

Lưu data

In [69]:
df.to_csv('data_cleaned.csv', index=False, encoding='utf-8-sig')

Kết nối sql

In [ ]:
import pyodbc
import pandas as pd
import sys

# Thông tin kết nối SQL Server với Windows Authentication
server = 'localhost' 
database = 'SocialMedia'  
driver = '{SQL Server}'

conn = None
conn_str = f'DRIVER={driver};SERVER={server};DATABASE={database};Trusted_Connection=True;'
try:
    print(f"Attempting to connect to: {conn_str}")
    conn = pyodbc.connect(conn_str)
    cursor = conn.cursor()
    print(f"Connection successful with server: {server}")
except pyodbc.Error as e:
    print(f"Failed with {conn_str}: {e}")

conn.commit()
conn.close()

# Kết nối lại với database vừa tạo
conn = pyodbc.connect(conn_str)
cursor = conn.cursor()
print("Reconnected to new database.")

# Kết nối lại với database vừa tạo
conn = pyodbc.connect(conn_str)
cursor = conn.cursor()
print("Reconnected to new database.")

# Tạo bảng Users với các cột mới
cursor.execute('''
    CREATE TABLE Users (
        user_id VARCHAR(255) PRIMARY KEY,
        city VARCHAR(255),
        country VARCHAR(255),
        region VARCHAR(255)
    )
''')

# Tạo bảng Brand (giữ nguyên, không thay đổi cột)
cursor.execute('''
    CREATE TABLE Brand (
        brand_id VARCHAR(255) PRIMARY KEY,
        brand_name VARCHAR(255)
    )
''')

# Tạo bảng Campaigns với các cột mới
cursor.execute('''
    CREATE TABLE Campaigns (
        campaign_id VARCHAR(255) PRIMARY KEY,
        brand_id VARCHAR(255),
        campaign_name VARCHAR(255),
        campaign_phase VARCHAR(255),
        product_name VARCHAR(255),
        FOREIGN KEY (brand_id) REFERENCES Brand(brand_id)
    )
''')

# Tạo bảng Post với các cột mới và khôi phục mối quan hệ
cursor.execute('''
    CREATE TABLE Post (
        post_id VARCHAR(255) PRIMARY KEY,
        user_id VARCHAR(255),
        platform VARCHAR(255),
        timestamp DATETIME,
        date DATE,
        time TIME,
        day_of_week VARCHAR(255),
        hashtags TEXT,
        emotion_type VARCHAR(255),
        sentiment_label VARCHAR(255),
        language VARCHAR(255),
        likes_count INT,
        shares_count INT,
        comments_count INT,
        impressions INT,
        mentions TEXT,
        campaign_id VARCHAR(255),
        text_content VARCHAR(MAX), 
        FOREIGN KEY (user_id) REFERENCES Users(user_id),
        FOREIGN KEY (campaign_id) REFERENCES Campaigns(campaign_id)
    )
''')

# Tạo bảng Post_Analytics với các cột mới và khôi phục mối quan hệ
cursor.execute('''
    CREATE TABLE Post_Analytics (
        post_id VARCHAR(255) PRIMARY KEY,
        buzz_change_rate FLOAT,
        engagement_rate FLOAT,
        toxicity_score FLOAT,
        user_engagement_growth FLOAT,
        FOREIGN KEY (post_id) REFERENCES Post(post_id)
    )
''')

Attempting to connect to: DRIVER={SQL Server};SERVER=localhost;DATABASE=SocialMedia;Trusted_Connection=True;
Connection successful with server: localhost
Reconnected to new database.
Reconnected to new database.


Insert data sql

In [71]:
# 1. Chèn dữ liệu vào Users
users_df = df[['user_id', 'city', 'country', 'region']].drop_duplicates()
for index, row in users_df.iterrows():
    cursor.execute("INSERT INTO Users (user_id, city, country, region) VALUES (?, ?, ?, ?)", 
                   (row['user_id'], row['city'] if pd.notna(row['city']) else None, row['country'] if pd.notna(row['country']) else None, row['region'] if pd.notna(row['region']) else None))
    conn.commit()

# 2. Chèn dữ liệu vào Brand
brand_df = df[['brand_name']].drop_duplicates()
for index, row in brand_df.iterrows():
    brand_id = get_brand_id(row['brand_name'])
    if brand_id is not None:
        cursor.execute("INSERT INTO Brand (brand_id, brand_name) VALUES (?, ?)", 
                       (brand_id, row['brand_name']))
        conn.commit()

# 3. Chèn dữ liệu vào Campaigns
campaigns_df = df[['campaign_name', 'campaign_phase', 'product_name', 'brand_name']].drop_duplicates(subset=['campaign_name'])
for index, row in campaigns_df.iterrows():
    campaign_id = get_campaign_id(row['campaign_name'])
    brand_id = get_brand_id(row['brand_name'])
    if campaign_id is not None and brand_id is not None:
        cursor.execute("IF NOT EXISTS (SELECT 1 FROM Campaigns WHERE campaign_id = ?) BEGIN INSERT INTO Campaigns (campaign_id, brand_id, campaign_name, campaign_phase, product_name) VALUES (?, ?, ?, ?, ?) END", 
                       (campaign_id, campaign_id, brand_id, row['campaign_name'], row['campaign_phase'] if pd.notna(row['campaign_phase']) else None, row['product_name'] if pd.notna(row['product_name']) else None))
        conn.commit()

# 4. Chèn dữ liệu vào Post (bao gồm text_content với string conversion for date and time)
post_df = df[[
    'post_id', 'user_id', 'platform', 'timestamp', 'date', 'time', 'day_of_week', 'hashtags',
    'emotion_type', 'sentiment_label', 'language', 'likes_count', 'shares_count', 'comments_count',
    'impressions', 'mentions', 'campaign_name', 'text_content'
]].drop_duplicates()
for index, row in post_df.iterrows():
    campaign_id = get_campaign_id(row['campaign_name'])
    if campaign_id is not None:
        try:
            # Chuyển đổi timestamp sang datetime Python
            timestamp_val = row['timestamp'] if pd.notna(row['timestamp']) else None
            if isinstance(timestamp_val, pd.Timestamp):
                timestamp_val = timestamp_val.to_pydatetime()

            # Chuyển đổi date và time sang chuỗi định dạng SQL
            date_val = row['date'].strftime('%Y-%m-%d') if pd.notna(row['date']) else None
            time_val = row['time'].strftime('%H:%M:%S') if pd.notna(row['time']) else None

            cursor.execute("INSERT INTO Post (post_id, user_id, platform, timestamp, date, time, day_of_week, hashtags, emotion_type, sentiment_label, language, likes_count, shares_count, comments_count, impressions, mentions, campaign_id, text_content) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)", 
                           (row['post_id'], row['user_id'], row['platform'], timestamp_val, date_val, time_val, row['day_of_week'] if pd.notna(row['day_of_week']) else None, row['hashtags'] if pd.notna(row['hashtags']) else None, row['emotion_type'] if pd.notna(row['emotion_type']) else None, row['sentiment_label'] if pd.notna(row['sentiment_label']) else None, row['language'] if pd.notna(row['language']) else None, row['likes_count'] if pd.notna(row['likes_count']) else None, row['shares_count'] if pd.notna(row['shares_count']) else None, row['comments_count'] if pd.notna(row['comments_count']) else None, row['impressions'] if pd.notna(row['impressions']) else None, row['mentions'] if pd.notna(row['mentions']) else None, campaign_id, row['text_content'] if pd.notna(row['text_content']) else None))
            conn.commit()
        except pyodbc.Error as e:
            print(f"Error at index {index}: {e}")
            print(f"Row data: {row}")
            continue

# 5. Chèn dữ liệu vào Post_Analytics
analytics_df = df[[
    'post_id', 'buzz_change_rate', 'engagement_rate', 'toxicity_score', 'user_engagement_growth'
]].drop_duplicates()
for index, row in analytics_df.iterrows():
    cursor.execute("INSERT INTO Post_Analytics (post_id, buzz_change_rate, engagement_rate, toxicity_score, user_engagement_growth) VALUES (?, ?, ?, ?, ?)", 
                   (row['post_id'], row['buzz_change_rate'] if pd.notna(row['buzz_change_rate']) else None, row['engagement_rate'] if pd.notna(row['engagement_rate']) else None, row['toxicity_score'] if pd.notna(row['toxicity_score']) else None, row['user_engagement_growth'] if pd.notna(row['user_engagement_growth']) else None))
    conn.commit()

# Đóng kết nối
conn.close()

print("Data inserted successfully!")

Data inserted successfully!
